In [ ]:
import h5py
import numpy as np
import umap
import plotly.graph_objs as go
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


# Ruta al archivo
ruta = "/content/DS_BAEHT_embeddings_ESM2(732) [1280 Dimensiones].h5"

# Inicializar los grupos
grupos = {'Ha': [], 'Hb': [], 'He': [], 'Ta': [], 'Tb': [], 'Te': []}

claves = []

# Abrir y recorrer todas las claves
with h5py.File(ruta, 'r') as archivo:
    for clave in archivo.keys():
        for clase in grupos.keys():
            if clave.startswith(clase):
                grupos[clase].append(archivo[clave][:])
                claves.append(clave)  # Guardar la clave correspondiente
                break

# Construir X e y
X = np.vstack([np.array(grupos[clase]) for clase in grupos])
y = [clase for clase in grupos for _ in grupos[clase]]

print(f"Total de embeddings cargados: {len(y)}")
print("Forma de X:", X.shape)

# Aplicar UMAP en 3D
reductor = umap.UMAP(
    n_components=3,
    metric='cosine',
    n_neighbors=60,
    min_dist=0.1,
    random_state=42
)
X_3d = reductor.fit_transform(X)

# Colores por clase
colores = {
    'Ha': '#000080',  # azul navy
    'Hb': '#DC143C',  # rojo
    'He': '#FFFF00',  # amarillo
    'Ta': '#FFA500',  # naranja
    'Tb': '#006400',  # verde
    'Te': '#800080'   # morado
}

# Crear lista de trazas (una por clase)
trazas = []
for clase in grupos.keys():
    indices = [i for i, etiqueta in enumerate(y) if etiqueta == clase]
    textos = [claves[i] for i in indices]  # Obtener los nombres para hover text
    traza = go.Scatter3d(
        x=X_3d[indices, 0],
        y=X_3d[indices, 1],
        z=X_3d[indices, 2],
        mode='markers',
        name=clase,
        text=textos,  # <-- Esto mostrará los nombres al pasar el cursor
        hoverinfo='text+name',
        marker=dict(
            size=5,
            color=colores[clase],
            opacity=0.7,
            line=dict(width=0.5, color='black')
        )
    )
    trazas.append(traza)

# Crear la figura
fig = go.Figure(data=trazas)

fig.update_layout(
    title='Embeddings de Hidrolasas y Transferasas de GH13 (UMAP 3D)',
    scene=dict(
        xaxis_title='UMAP 1',
        yaxis_title='UMAP 2',
        zaxis_title='UMAP 3'
    ),
    legend_title='CAZymes',
    margin=dict(l=0, r=0, b=0, t=30),
    width=900,
    height=700
)

# Mostrar figura
fig.show()

# Guardar la figura como archivo HTML
fig.write_html("umap_3d_embeddings.html")

print("Visualización guardada como 'UMAP3D_732_Embeddings.html'")


Total de embeddings cargados: 724
Forma de X: (724, 1280)


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Visualización guardada como 'UMAP3D_732_Embeddings.html'
